In [6]:
import ee
import geemap
ee.Authenticate()
ee.Initialize(project='sds-analysis-493219')

print("Connected!")

Connected!


In [ ]:
# ======================================================
# 0) ROI: Dubai using geoBoundaries ADM2
# ======================================================
adm2 = ee.FeatureCollection("WM/geoLab/geoBoundaries/600/ADM2")

dubaiFC = (adm2
    .filter(ee.Filter.eq('shapeGroup', 'ARE'))   # UAE ISO3 = ARE
    .filter(ee.Filter.eq('shapeName', 'Dubai')))

roi = dubaiFC.geometry()

print('Dubai features found:', dubaiFC.size().getInfo())

In [ ]:

# ======================================================
# 1) Load datasets
# ======================================================
mod09 = ee.ImageCollection("MODIS/061/MOD09GA")
lc    = ee.ImageCollection("MODIS/061/MCD12Q1")
dem   = ee.Image("USGS/SRTMGL1_003")

# ======================================================
# 2) Helpers: QA + scaling + indices
# ======================================================
def scale_and_mask_MOD09GA(img):
    # Select SR bands and scale to reflectance-like values
    b = img.select([
        "sur_refl_b01", "sur_refl_b02", "sur_refl_b03", "sur_refl_b04",
        "sur_refl_b05", "sur_refl_b06", "sur_refl_b07"
    ]).multiply(0.0001)

    # Simple cloud screening using state_1km bits 0-1
    # 0=clear, 1=cloudy, 2=mixed, 3=not set/assumed clear
    state = img.select("state_1km")
    cloud_state = state.bitwiseAnd(3)  # bits 0-1
    clear = cloud_state.eq(0).Or(cloud_state.eq(3))

    return (img.addBands(b, None, True)
               .updateMask(clear)
               .copyProperties(img, img.propertyNames()))


def add_indices(img):
    B1 = img.select("sur_refl_b01")
    B2 = img.select("sur_refl_b02")
    B3 = img.select("sur_refl_b03")
    B4 = img.select("sur_refl_b04")
    B6 = img.select("sur_refl_b06")
    B7 = img.select("sur_refl_b07")

    # NDDI = (B7 - B3) / (B7 + B3)
    nddi = B7.subtract(B3).divide(B7.add(B3)).rename("NDDI")

    # MBSCI = (B4 + B1) / (B6 + B7)
    mbsci = B4.add(B1).divide(B6.add(B7)).rename("MBSCI")

    # NDVI
    ndvi = B2.subtract(B1).divide(B2.add(B1)).rename("NDVI")

    # NDWI (McFeeters-style)
    ndwi = B4.subtract(B2).divide(B4.add(B2)).rename("NDWI")

    # NDSI = (B4 - B6) / (B4 + B6)
    ndsi = B4.subtract(B6).divide(B4.add(B6)).rename("NDSI")

    # MBWI = 2*B4 - B1 - B2 - B6 - B7
    mbwi = (B4.multiply(2)
              .subtract(B1)
              .subtract(B2)
              .subtract(B6)
              .subtract(B7)
              .rename("MBWI"))

    return img.addBands([nddi, mbsci, ndvi, ndwi, ndsi, mbwi])



Dubai features found: 1


In [ ]:
# ======================================================
# 3) Build daily collection for study period
# ======================================================
start = "2022-01-01"
end   = "2022-12-31"

ic = (mod09
      .filterDate(start, end)
      .filterBounds(roi)
      .map(scale_and_mask_MOD09GA)
      .map(add_indices))

# ======================================================
# 4) Filter to a single day and clip to ROI
# ======================================================
date = ee.Date('2022-04-18')

day_col = ic.filterDate(date, date.advance(1, 'day'))
print('Images found for that day:', day_col.size().getInfo())

img0 = ee.Image(day_col.sort('system:time_start').first()).clip(roi)

# ======================================================
# 5) Visualize with geemap
# ======================================================
Map = geemap.Map()
Map.centerObject(roi, 9)

# ROI boundary
Map.addLayer(roi, {'color': 'grey'}, 'Dubai boundary')

# True color
Map.addLayer(
    img0.select(["sur_refl_b02", "sur_refl_b04", "sur_refl_b03"]),
    {'min': 0.0, 'max': 0.4},
    'True color-ish'
)

Map.addLayer(
    img0.select(["sur_refl_b02", "sur_refl_b04", "sur_refl_b03"]),
    {'min': 0.0, 'max': 0.4, 'gamma': 1.4},
    'True color (MODIS)'
)

# Indices
Map.addLayer(
    img0.select('NDVI'),
    {'min': -0.2, 'max': 0.8, 'palette': ['brown', 'yellow', 'green']},
    'NDVI'
)

Map.addLayer(
    img0.select('NDSI'),
    {'min': -1, 'max': 1, 'palette': ['purple', 'white', 'cyan']},
    'NDSI'
)

Map.addLayer(
    img0.select('MBWI'),
    {'min': -0.5, 'max': 0.5, 'palette': ['brown', 'white', 'blue']},
    'MBWI'
)



Images found for that day: 1


Map(center=[24.942870909915, 55.3881772570556], controls=(WidgetControl(options=['position', 'transparent_bg']…

In [ ]:
# Display the map
Map

In [9]:
from IPython.display import display
display(Map)


Map(center=[24.942870909915, 55.3881772570556], controls=(WidgetControl(options=['position', 'transparent_bg']…